# Field Mapping Configuration

This notebook scans your actual JSON files, figures out what fields exist,
matches them to extractable email fields, and configures the pipeline.

**Run each cell in order.** The workflow is:
1. **Cell 1** — Scan your JSONs and auto-generate mapping
2. **Cell 2** — Interactive GUI: walk through each field, confirm/adjust/skip
3. **Cell 3** — Review & save your configuration
4. **Cell 4** — (Optional) Add new custom fields
5. **Cell 5** — (Optional) Remove fields

Your configuration is saved to `data/field_mapping.json` and automatically used
by the extraction GUI and web server.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1: SCAN JSONs & AUTO-GENERATE MAPPING
# ═══════════════════════════════════════════════════════════════════════
# Reads every JSON file in data/, discovers the fields, and matches
# them to things we know how to extract from CLO emails.
# ═══════════════════════════════════════════════════════════════════════

import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from backend.field_mapping import (
    load_mapping, save_mapping, scan_stores, DEFAULT_MAPPING,
    ALL_STORES, MAPPING_FILE,
)
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import json
import copy

# Scan the actual JSON files
mapping = scan_stores()
field_names = list(mapping['fields'].keys())

extractable = [n for n, c in mapping['fields'].items() if c.get('extract', True)]
not_extractable = [n for n, c in mapping['fields'].items() if not c.get('extract', True)]

print(f'Scanned JSON stores in data/')
print(f'  Found {len(field_names)} fields total')
print()

if extractable:
    print(f'Auto-matched to email extraction ({len(extractable)}):')
    for name in extractable:
        cfg = mapping['fields'][name]
        store = cfg.get('store') or '(routing)'
        sf = cfg.get('store_field') or '—'
        print(f'  ✅ {name:40s} -> {store}.{sf}')

if not_extractable:
    print(f'\nFields in your JSONs not yet linked to extraction ({len(not_extractable)}):')
    for name in not_extractable:
        cfg = mapping['fields'][name]
        store = cfg.get('store') or '?'
        sf = cfg.get('store_field') or name
        print(f'  ❓ {sf:40s} (in {store}.json)')
    print()
    print('  These are marked as SKIP by default. In Cell 2, you can enable')
    print('  any of them and write extraction rules in your custom_rules module.')

if os.path.isfile(MAPPING_FILE):
    print(f'\nNote: existing config at {MAPPING_FILE} will be replaced when you save in Cell 3.')

print(f'\nRun Cell 2 to walk through each field.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2: INTERACTIVE FIELD CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════
#   Walk through each field one by one.
#   For each field you can:
#     - Keep it as-is (matches your JSON)
#     - Change which store it maps to
#     - Change the field name in the store
#     - Mark it as SKIP (won't be extracted, always blank)
#     - Change the display label
# ═══════════════════════════════════════════════════════════════════════

# ── State ──
# Keep a copy of the scanned mapping as the "baseline" for Keep Default
_scanned_mapping = copy.deepcopy(mapping)
_cfg_state = {'idx': 0, 'mapping': copy.deepcopy(mapping)}
_field_list = list(_cfg_state['mapping']['fields'].keys())

# ── Header ──
header = widgets.HTML(layout=widgets.Layout(padding='8px'))

# ── Field info (read-only) ──
info_html = widgets.HTML(layout=widgets.Layout(padding='4px 8px'))

# ── Editable controls ──
store_options = ['(none — routing only)'] + ALL_STORES
w_store = widgets.Dropdown(
    options=store_options,
    description='Target Store:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px'),
)
w_store_field = widgets.Text(
    description='Store Field:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px'),
    placeholder='field name in the JSON store',
)
w_label = widgets.Text(
    description='GUI Label:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px'),
)
w_form_type = widgets.Dropdown(
    options=['text', 'textarea', 'float', 'dropdown'],
    description='Widget Type:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px'),
)
w_extract = widgets.Checkbox(
    description='Extract this field (uncheck to SKIP — always blank)',
    style={'description_width': 'initial'},
    indent=False,
    layout=widgets.Layout(width='450px'),
)
w_notes = widgets.Text(
    description='Notes:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px'),
    placeholder='optional notes for your reference',
)

# ── Navigation buttons ──
btn_prev = widgets.Button(description='< Previous', layout=widgets.Layout(width='120px'))
btn_next = widgets.Button(description='Next >', button_style='primary', layout=widgets.Layout(width='120px'))
btn_skip = widgets.Button(description='Mark as SKIP', button_style='warning', layout=widgets.Layout(width='140px'))
btn_keep = widgets.Button(description='Keep As-Is', button_style='info', layout=widgets.Layout(width='140px'))
nav_bar = widgets.HBox([btn_prev, btn_next, widgets.HTML(value='&nbsp;&nbsp;&nbsp;'), btn_skip, btn_keep])

# ── Log ──
log = widgets.Output(layout=widgets.Layout(max_height='150px', overflow_y='auto', border='1px solid #444', padding='4px'))

# ── Layout ──
form = widgets.VBox([
    header,
    info_html,
    widgets.HTML(value='<hr style="margin:4px 0;">'),
    w_extract,
    w_store,
    w_store_field,
    w_label,
    w_form_type,
    w_notes,
    widgets.HTML(value='<hr style="margin:4px 0;">'),
    nav_bar,
    log,
], layout=widgets.Layout(padding='10px', border='2px solid #555', width='600px'))


def _save_current():
    """Save current widget values into the mapping dict."""
    idx = _cfg_state['idx']
    if idx >= len(_field_list):
        return
    name = _field_list[idx]
    cfg = _cfg_state['mapping']['fields'][name]
    cfg['extract'] = w_extract.value
    store_val = w_store.value
    cfg['store'] = None if store_val == store_options[0] else store_val
    cfg['store_field'] = w_store_field.value or None
    cfg['label'] = w_label.value
    cfg['form_type'] = w_form_type.value
    cfg['notes'] = w_notes.value


def _load_field(idx):
    """Load field at index into the widgets."""
    if idx < 0 or idx >= len(_field_list):
        return
    _cfg_state['idx'] = idx
    name = _field_list[idx]
    cfg = _cfg_state['mapping']['fields'][name]

    header.value = (
        f"<b style='font-size:15px;'>Field {idx + 1} of {len(_field_list)}: "
        f"<code style='color:#4fc3f7;'>{name}</code></b>"
    )

    # Show where this field was discovered from
    scanned_cfg = _scanned_mapping['fields'].get(name, {})
    src_store = scanned_cfg.get('store', '—')
    src_field = scanned_cfg.get('store_field', '—')
    extractable = '✅ extractable' if scanned_cfg.get('extract', False) else '❓ not yet linked'
    info_html.value = (
        f"<span style='color:#aaa;'>Found in: "
        f"<b>{src_store}</b>.json as <b>{src_field}</b> — {extractable}</span>"
    )

    w_extract.value = cfg.get('extract', True)
    store_val = cfg.get('store')
    w_store.value = store_val if store_val in ALL_STORES else store_options[0]
    w_store_field.value = cfg.get('store_field') or ''
    w_label.value = cfg.get('label', name)
    w_form_type.value = cfg.get('form_type', 'text')
    w_notes.value = cfg.get('notes', '')

    btn_prev.disabled = (idx == 0)
    btn_next.description = 'Next >' if idx < len(_field_list) - 1 else 'Done'


def _advance():
    """Move to next field or show done message."""
    if _cfg_state['idx'] < len(_field_list) - 1:
        _load_field(_cfg_state['idx'] + 1)
    else:
        header.value = "<b style='font-size:15px; color:#66bb6a;'>All fields configured! Run Cell 3 to review & save.</b>"
        with log:
            print('\nDone! Run Cell 3 to review and save your configuration.')


def _on_next(btn):
    _save_current()
    idx = _cfg_state['idx']
    name = _field_list[idx]
    cfg = _cfg_state['mapping']['fields'][name]
    status = 'SKIP' if not cfg['extract'] else (cfg.get('store') or 'routing')
    with log:
        print(f'  {name} -> {status}' + (f" (as '{cfg.get(\"store_field\", name)}')\" if cfg['extract'] and cfg.get('store') else ''))
    _advance()


def _on_prev(btn):
    _save_current()
    idx = _cfg_state['idx']
    if idx > 0:
        _load_field(idx - 1)


def _on_skip_field(btn):
    """Quick-mark current field as skip and advance."""
    w_extract.value = False
    _save_current()
    name = _field_list[_cfg_state['idx']]
    with log:
        print(f'  {name} -> SKIP (will always be blank)')
    _advance()


def _on_keep(btn):
    """Keep the scanned mapping for this field and advance."""
    name = _field_list[_cfg_state['idx']]
    # Restore from what scan_stores() found (not hardcoded defaults)
    if name in _scanned_mapping['fields']:
        _cfg_state['mapping']['fields'][name] = copy.deepcopy(_scanned_mapping['fields'][name])
    cfg = _cfg_state['mapping']['fields'][name]
    status = 'SKIP' if not cfg.get('extract', True) else (cfg.get('store') or 'routing')
    with log:
        print(f'  {name} -> {status} (kept as-is)')
    _advance()


btn_next.on_click(_on_next)
btn_prev.on_click(_on_prev)
btn_skip.on_click(_on_skip_field)
btn_keep.on_click(_on_keep)

# Start at field 0
_load_field(0)
with log:
    print('Walk through each field. Use Next/Previous to navigate.')
    print('Mark as SKIP = field will not be extracted (always blank).')
    print('Keep As-Is = accept what was discovered from your JSONs.')
    print()

display(form)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3: REVIEW & SAVE
# ═══════════════════════════════════════════════════════════════════════
#   Shows a summary of your configuration and saves it to disk.
# ═══════════════════════════════════════════════════════════════════════

# Make sure last field is saved
_save_current()
final_mapping = _cfg_state['mapping']

# ── Summary ──
active = [n for n, c in final_mapping['fields'].items() if c.get('extract', True)]
skipped = [n for n, c in final_mapping['fields'].items() if not c.get('extract', True)]

print('=' * 65)
print('FIELD MAPPING SUMMARY')
print('=' * 65)
print()
print(f'Active fields ({len(active)}):')
print('-' * 65)
for name in active:
    cfg = final_mapping['fields'][name]
    store = cfg.get('store') or '(routing)'
    sf = cfg.get('store_field') or '—'
    label = cfg.get('label', name)
    print(f'  {name:40s} -> {store}.{sf}')

if skipped:
    print()
    print(f'Skipped fields ({len(skipped)}) — will always be blank:')
    print('-' * 65)
    for name in skipped:
        print(f'  {name}')

# ── Group by store ──
print()
print('By store:')
print('-' * 65)
for store_name in ALL_STORES:
    fields = [(n, c.get('store_field', n)) for n, c in final_mapping['fields'].items()
              if c.get('store') == store_name and c.get('extract', True)]
    if fields:
        print(f'  {store_name}.json:')
        for ext_name, store_field in fields:
            arrow = f' (as "{store_field}")' if store_field != ext_name else ''
            print(f'    {ext_name}{arrow}')

print()
print('=' * 65)

# ── Save ──
save_mapping(final_mapping)
print(f'\nSaved to {MAPPING_FILE}')
print('This mapping will be used automatically by the extraction GUI and web server.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4: ADD NEW CUSTOM FIELDS (optional)
# ═══════════════════════════════════════════════════════════════════════
#   Use this to add extraction fields that aren't in the defaults.
#   For example, if your JSON has a "rating_agency" field that the
#   default extractor doesn't know about.
#
#   After adding, re-run Cell 3 to review & save.
# ═══════════════════════════════════════════════════════════════════════

add_log = widgets.Output(layout=widgets.Layout(max_height='200px', overflow_y='auto', border='1px solid #444', padding='4px'))

w_new_name = widgets.Text(description='Field Name:', placeholder='e.g. rating_agency', style={'description_width': '120px'}, layout=widgets.Layout(width='400px'))
w_new_label = widgets.Text(description='GUI Label:', placeholder='e.g. Rating Agency', style={'description_width': '120px'}, layout=widgets.Layout(width='400px'))
w_new_store = widgets.Dropdown(options=store_options, description='Target Store:', style={'description_width': '120px'}, layout=widgets.Layout(width='400px'))
w_new_sf = widgets.Text(description='Store Field:', placeholder='field name in JSON (default = same as field name)', style={'description_width': '120px'}, layout=widgets.Layout(width='500px'))
w_new_type = widgets.Dropdown(options=['text', 'textarea', 'float', 'dropdown'], description='Widget Type:', style={'description_width': '120px'}, layout=widgets.Layout(width='400px'))
w_new_notes = widgets.Text(description='Notes:', style={'description_width': '120px'}, layout=widgets.Layout(width='500px'))
btn_add = widgets.Button(description='Add Field', button_style='success', layout=widgets.Layout(width='120px'))


def _on_add(btn):
    name = w_new_name.value.strip()
    if not name:
        with add_log:
            print('Field name is required.')
        return
    if name in _cfg_state['mapping']['fields']:
        with add_log:
            print(f'Field "{name}" already exists. Edit it in Cell 2 instead.')
        return

    store_val = w_new_store.value
    new_cfg = {
        'label': w_new_label.value.strip() or name,
        'store': None if store_val == store_options[0] else store_val,
        'store_field': w_new_sf.value.strip() or name,
        'extract': True,
        'form_type': w_new_type.value,
        'notes': w_new_notes.value.strip(),
    }
    _cfg_state['mapping']['fields'][name] = new_cfg
    _field_list.append(name)

    with add_log:
        store_display = new_cfg['store'] or '(routing)'
        print(f'Added: {name} -> {store_display}.{new_cfg["store_field"]}')

    # Clear inputs
    w_new_name.value = ''
    w_new_label.value = ''
    w_new_sf.value = ''
    w_new_notes.value = ''


btn_add.on_click(_on_add)

add_form = widgets.VBox([
    widgets.HTML(value="<b>Add a new extraction field:</b>"),
    w_new_name, w_new_label, w_new_store, w_new_sf, w_new_type, w_new_notes,
    btn_add,
    widgets.HTML(value="<br><i style='color:#aaa;'>After adding fields, re-run Cell 3 to review & save.</i>"),
    add_log,
], layout=widgets.Layout(padding='10px', border='2px solid #555', width='600px'))

display(add_form)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5: REMOVE FIELDS (optional)
# ═══════════════════════════════════════════════════════════════════════
#   Remove a field entirely from the mapping. Different from SKIP:
#   - SKIP keeps the field in config but never extracts it
#   - REMOVE deletes it from the config entirely
#
#   After removing, re-run Cell 3 to review & save.
# ═══════════════════════════════════════════════════════════════════════

rm_log = widgets.Output(layout=widgets.Layout(max_height='150px', overflow_y='auto', border='1px solid #444', padding='4px'))
current_fields = list(_cfg_state['mapping']['fields'].keys())
w_rm_field = widgets.Dropdown(options=current_fields, description='Field:', style={'description_width': '80px'}, layout=widgets.Layout(width='400px'))
btn_rm = widgets.Button(description='Remove Field', button_style='danger', layout=widgets.Layout(width='140px'))


def _on_remove(btn):
    name = w_rm_field.value
    if name in _cfg_state['mapping']['fields']:
        del _cfg_state['mapping']['fields'][name]
        if name in _field_list:
            _field_list.remove(name)
        remaining = list(_cfg_state['mapping']['fields'].keys())
        w_rm_field.options = remaining
        with rm_log:
            print(f'Removed: {name}')
    else:
        with rm_log:
            print(f'Field "{name}" not found.')


btn_rm.on_click(_on_remove)

rm_form = widgets.VBox([
    widgets.HTML(value="<b>Remove a field from the mapping:</b>"),
    w_rm_field, btn_rm,
    widgets.HTML(value="<br><i style='color:#aaa;'>After removing fields, re-run Cell 3 to review & save.</i>"),
    rm_log,
], layout=widgets.Layout(padding='10px', border='2px solid #555', width='600px'))

display(rm_form)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6: RESET TO DEFAULTS (emergency)
# ═══════════════════════════════════════════════════════════════════════
#   Uncomment and run to reset the mapping to defaults.
#   WARNING: This overwrites your saved configuration!
# ═══════════════════════════════════════════════════════════════════════

# import os
# from backend.field_mapping import MAPPING_FILE, DEFAULT_MAPPING, save_mapping
# save_mapping(DEFAULT_MAPPING)
# print(f'Reset to defaults. Re-run Cell 1 to reload.')